# Bronze Layer - Raw Data Ingestion
### Egyptian Investment Intelligence Platform

Goal: Ingest all raw data sources into Iceberg tables on S3 with no transformations.

| Table | Source | Approx Rows |
|---|---|---|
| bronze.stocks_eod | batch_eod_all_stocks.csv | 1230 |
| bronze.egx30_index | EGX30_index.csv | 123 |
| bronze.fundamentals | fundamentals_all.csv | 10 |
| bronze.live_quotes | live_quotes_all.csv | 10 |
| bronze.gold_silver_prices | authority_prices.csv | 80638 |
| bronze.currency_rates | currency_rates.csv | 348 |
| bronze.spot_prices | spot_prices.csv | 8 |
| bronze.real_estate | data.json (Bayut) | 107 |
| bronze.real_estate_enriched | data_enriched.json (PropertyFinder) | 300 |

Rule: Use s3a:// for reading raw files (Spark/Hadoop). Use s3:// for Iceberg table locations (S3FileIO).

## Step 1 - Upload Raw Files to S3
> Run once. Skip if files are already uploaded.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"], check=True)

import boto3, os

s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name='eu-north-1'
)

BUCKET     = 'my-icebergdatalake'
SOURCE_DIR = '/home/docker/source_data'

files_to_upload = {
    'batch_eod_all_stocks.csv' : 'raw/batch_eod_all_stocks.csv',
    'EGX30_index.csv'          : 'raw/EGX30_index.csv',
    'fundamentals_all.csv'     : 'raw/fundamentals_all.csv',
    'live_quotes_all.csv'      : 'raw/live_quotes_all.csv',
    'authority_prices.csv'     : 'raw/authority_prices.csv',
    'currency_rates.csv'       : 'raw/currency_rates.csv',
    'spot_prices.csv'          : 'raw/spot_prices.csv',
    'data.json'                : 'raw/data.json',
    'data_enriched.json'       : 'raw/data_enriched.json',
}

for local_name, s3_key in files_to_upload.items():
    local_path = os.path.join(SOURCE_DIR, local_name)
    if os.path.exists(local_path):
        s3.upload_file(local_path, BUCKET, s3_key)
        print(f'Uploaded: {local_name} -> s3://{BUCKET}/{s3_key}')
    else:
        print(f'File not found: {local_path}')

print('All files uploaded to S3.')

## Step 2 - Start Spark Session

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp

spark = SparkSession.builder \
    .appName('Bronze_Layer_Ingestion') \
    .config('spark.jars.packages',
            'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.0,'
            'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
            'software.amazon.awssdk:url-connection-client:2.20.18') \
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
            'org.projectnessie.spark.extensions.NessieSparkSessionExtensions') \
    .config('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog') \
    .config('spark.sql.catalog.nessie.uri', 'http://nessie:19120/api/v1') \
    .config('spark.sql.catalog.nessie.ref', 'main') \
    .config('spark.sql.catalog.nessie.warehouse', 's3://my-icebergdatalake/iceberg-warehouse/') \
    .config('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'com.amazonaws.auth.EnvironmentVariableCredentialsProvider') \
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID']) \
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY']) \
    .config('spark.hadoop.fs.s3a.endpoint.region', 'eu-north-1') \
    .config('spark.driver.host', 'localhost') \
    .config('spark.driver.bindAddress', '0.0.0.0') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark Session started successfully')

## Step 3 - Create Namespaces and Helper Function
> Skip if already created in a previous session.

In [ ]:
from pyspark.sql.functions import lit, current_timestamp

spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.bronze')
spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.silver')
spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.gold')
print('Namespaces ready')

def add_metadata(df, source_file):
    return df \
        .withColumn('_ingested_at', current_timestamp()) \
        .withColumn('_source_file', lit(source_file))

print('Helper function defined')

## Table 1 - bronze.stocks_eod
> Source: batch_eod_all_stocks.csv - All 10 EGX stocks daily OHLCV combined

In [ ]:
df_stocks_eod = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/batch_eod_all_stocks.csv')

df_stocks_eod = add_metadata(df_stocks_eod, 'batch_eod_all_stocks.csv')

df_stocks_eod.writeTo('nessie.bronze.stocks_eod') \
    .tableProperty('write.target-file-size-bytes', '134217728') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/stocks_eod') \
    .createOrReplace()

print(f'stocks_eod - {df_stocks_eod.count()} rows ingested')
df_stocks_eod.printSchema()
df_stocks_eod.show(3)

## Table 2 - bronze.egx30_index
> Source: EGX30_index.csv - EGX30 benchmark index daily prices

In [ ]:
df_egx30 = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/EGX30_index.csv')

df_egx30 = add_metadata(df_egx30, 'EGX30_index.csv')

df_egx30.writeTo('nessie.bronze.egx30_index') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/egx30_index') \
    .createOrReplace()

print(f'egx30_index - {df_egx30.count()} rows ingested')
df_egx30.printSchema()
df_egx30.show(3)

## Table 3 - bronze.fundamentals
> Source: fundamentals_all.csv - Company financials: P/E, MarketCap, Beta, EPS

In [ ]:
df_fundamentals = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/fundamentals_all.csv')

df_fundamentals = add_metadata(df_fundamentals, 'fundamentals_all.csv')

df_fundamentals.writeTo('nessie.bronze.fundamentals') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/fundamentals') \
    .createOrReplace()

print(f'fundamentals - {df_fundamentals.count()} rows ingested')
df_fundamentals.printSchema()
df_fundamentals.show()

## Table 4 - bronze.live_quotes
> Source: live_quotes_all.csv - Real-time stock quotes with change %

In [ ]:
df_live = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/live_quotes_all.csv')

df_live = add_metadata(df_live, 'live_quotes_all.csv')

df_live.writeTo('nessie.bronze.live_quotes') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/live_quotes') \
    .createOrReplace()

print(f'live_quotes - {df_live.count()} rows ingested')
df_live.printSchema()
df_live.show()

## Table 5 - bronze.gold_silver_prices
> Source: authority_prices.csv - Historical LBMA gold and silver prices (~80,638 rows)

In [ ]:
df_metals = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/authority_prices.csv')

df_metals = add_metadata(df_metals, 'authority_prices.csv')

df_metals.writeTo('nessie.bronze.gold_silver_prices') \
    .tableProperty('write.target-file-size-bytes', '134217728') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/gold_silver_prices') \
    .createOrReplace()

print(f'gold_silver_prices - {df_metals.count()} rows ingested')
df_metals.printSchema()
df_metals.show(3)

## Table 6 - bronze.currency_rates
> Source: currency_rates.csv - Exchange rates (~348 rows)

In [ ]:
df_currency = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/currency_rates.csv')

df_currency = add_metadata(df_currency, 'currency_rates.csv')

df_currency.writeTo('nessie.bronze.currency_rates') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/currency_rates') \
    .createOrReplace()

print(f'currency_rates - {df_currency.count()} rows ingested')
df_currency.printSchema()
df_currency.show(3)

## Table 7 - bronze.spot_prices
> Source: spot_prices.csv - Current gold and silver spot prices with bid/ask (8 rows)

In [ ]:
df_spot = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('s3a://my-icebergdatalake/raw/spot_prices.csv')

df_spot = add_metadata(df_spot, 'spot_prices.csv')

df_spot.writeTo('nessie.bronze.spot_prices') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/spot_prices') \
    .createOrReplace()

print(f'spot_prices - {df_spot.count()} rows ingested')
df_spot.printSchema()
df_spot.show()

## Table 8 - bronze.real_estate
> Source: data.json - Real estate listings from Bayut.eg (107 rows)

In [ ]:
df_real_estate = spark.read \
    .option('multiline', True) \
    .json('s3a://my-icebergdatalake/raw/data.json')

df_real_estate = add_metadata(df_real_estate, 'data.json')

df_real_estate.writeTo('nessie.bronze.real_estate') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/real_estate') \
    .createOrReplace()

print(f'real_estate - {df_real_estate.count()} rows ingested')
df_real_estate.printSchema()
df_real_estate.show(3, truncate=False)

## Table 9 - bronze.real_estate_enriched
> Source: data_enriched.json - Enriched listings from PropertyFinder.eg (300 rows)

In [ ]:
df_re_enriched = spark.read \
    .option('multiline', True) \
    .json('s3a://my-icebergdatalake/raw/data_enriched.json')

df_re_enriched = add_metadata(df_re_enriched, 'data_enriched.json')

df_re_enriched.writeTo('nessie.bronze.real_estate_enriched') \
    .tableProperty('location', 's3://my-icebergdatalake/bronze/real_estate_enriched') \
    .createOrReplace()

print(f'real_estate_enriched - {df_re_enriched.count()} rows ingested')
df_re_enriched.printSchema()
df_re_enriched.show(3, truncate=False)

## Final Verification - All Bronze Tables

In [ ]:
print('=' * 55)
print('  BRONZE LAYER - FINAL VERIFICATION')
print('=' * 55)

bronze_tables = [
    'nessie.bronze.stocks_eod',
    'nessie.bronze.egx30_index',
    'nessie.bronze.fundamentals',
    'nessie.bronze.live_quotes',
    'nessie.bronze.gold_silver_prices',
    'nessie.bronze.currency_rates',
    'nessie.bronze.spot_prices',
    'nessie.bronze.real_estate',
    'nessie.bronze.real_estate_enriched',
]

total = 0
for table in bronze_tables:
    count = spark.table(table).count()
    total += count
    name = table.split('.')[-1]
    print(f'   {name:<40} {count:>7} rows')

print('=' * 55)
print(f'   TOTAL ROWS INGESTED: {total}')
print('=' * 55)
print('Bronze Layer complete.')
print('All tables stored at : s3://my-icebergdatalake/bronze/')
print('All tables tracked by: Nessie catalog -> nessie.bronze.*')
print('Next step: Run 02_silver_layer.ipynb')